# Lab 15 solution: a named tool for the country question, and a way to start over

This notebook solves the two Challenge tasks of lab 15. It repeats the setup it needs (the config cell, the read-only
database, the four named lookups and the desk agent, as in the lab and `app.py`), so it runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import difflib
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # a progress-bar notice from a Gradio dependency
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"                   # Gradio sends usage statistics unless told not to

import gradio as gr
from agents import Agent, ModelSettings, Runner, SQLiteSession, function_tool, set_tracing_disabled
from openai.types.shared import Reasoning

set_tracing_disabled(True)
LAB = ROOT / "15_Support_Desk_Agent"
OUTPUTS = LAB / "outputs"          # files this notebook writes while it runs; git ignores this folder
OUTPUTS.mkdir(exist_ok=True)

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)
ALL_NAMES = [row[0] for row in conn.execute("SELECT FirstName || ' ' || LastName FROM customers")]

The four named lookups and the desk agent, as in lab 15 Part A:

In [3]:
def find_customer(name: str) -> dict:
    """Find a Chinook customer by first and last name. Returns the exact matches, each with its customer_id;
    when there are none, up to three close names to try."""
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    matches = [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]
    if matches:
        return {"matches": matches}
    return {"matches": [], "close_names": difflib.get_close_matches(name, ALL_NAMES, n=3, cutoff=0.8)}


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their totals in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep assigned to a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}


def get_last_invoice(customer_id: int) -> dict:
    """Return a customer's most recent invoice: its number, date and total in US dollars."""
    row = conn.execute(
        "SELECT InvoiceId, date(InvoiceDate), Total FROM invoices WHERE CustomerId = ? "
        "ORDER BY InvoiceDate DESC, InvoiceId DESC LIMIT 1", (customer_id,)).fetchone()
    if row is None:
        return {"customer_id": customer_id, "invoice_id": None}
    return {"customer_id": customer_id, "invoice_id": row[0], "date": row[1], "total": row[2]}


INSTRUCTIONS = ("You help the support reps of a music shop. Look up every fact with the tools; never answer from "
                "memory. Answer in one or two sentences.")
desk_agent = Agent(
    name="Chinook desk",
    instructions=INSTRUCTIONS,
    model=MODEL,
    tools=[function_tool(f) for f in (find_customer, get_invoices, get_support_rep, get_last_invoice)],
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT)),
)
print([t.name for t in desk_agent.tools])

['find_customer', 'get_invoices', 'get_support_rep', 'get_last_invoice']


## Task 1: Answer the country question with a named tool

In lab 15 the named tools declined *"Which billing country has spent the most at the shop, and how much?"*: no tool
adds up invoices by country. A fifth named tool runs that one query, with the number of countries as its only
argument. The `?` keeps the model's `limit` a value, never a piece of SQL.

In [4]:
def top_billing_countries(limit: int) -> list:
    """List the billing countries whose invoices add up to the most, largest first, each with its total in US
    dollars."""
    rows = conn.execute(
        "SELECT BillingCountry, ROUND(SUM(Total), 2) FROM invoices "
        "GROUP BY BillingCountry ORDER BY 2 DESC LIMIT ?", (limit,)).fetchall()
    return [{"country": r[0], "total": r[1]} for r in rows]


print(top_billing_countries(3))

[{'country': 'USA', 'total': 523.06}, {'country': 'Canada', 'total': 303.96}, {'country': 'France', 'total': 195.1}]


The USA first with 523.06, as lab 15's own query gave. `clone` copies the agent and replaces the fields we name; the
copy keeps the instructions, the model and the model settings:

In [5]:
desk_plus = desk_agent.clone(tools=desk_agent.tools + [function_tool(top_billing_countries)])
print(desk_plus.name, "|", [t.name for t in desk_plus.tools], "|", desk_plus.model_settings.reasoning)

Chinook desk | ['find_customer', 'get_invoices', 'get_support_rep', 'get_last_invoice', 'top_billing_countries'] | Reasoning(context=None, effort='none', generate_summary=None, mode=None, summary=None)


In [6]:
COUNTRY = "Which billing country has spent the most at the shop, and how much?"
right_count = 0
for run in range(1, 4):
    result = await Runner.run(desk_plus, COUNTRY)
    calls = [f"{i.raw_item.name}{i.raw_item.arguments}" for i in result.new_items if i.type == "tool_call_item"]
    right = "USA" in result.final_output and "523.06" in result.final_output
    right_count += right
    print(f"run {run} {'✅' if right else '❌'} calls: {calls}")
    print("       ", result.final_output)
print(f"right in {right_count} of 3")

run 1 ✅ calls: ['top_billing_countries{"limit":1}']
        The USA has spent the most, with a total of **$523.06**.


run 2 ✅ calls: ['top_billing_countries{"limit":1}']
        The USA has spent the most, with a total of **$523.06**.


run 3 ✅ calls: ['top_billing_countries{"limit":1}']
        The USA has spent the most, with a total of **$523.06**.
right in 3 of 3


Right in 3 of 3, each with one call to the new tool, `top_billing_countries` with `limit` 1, and the reply
*"The USA has spent the most, with a total of $523.06."* The tool's description told the model which tool adds up
countries, and the model chose how many to ask for.

The delete request still has nowhere to go:

In [7]:
result = await Runner.run(desk_plus, "Delete Helena Holý's last invoice, please.")
print([f"{i.raw_item.name}{i.raw_item.arguments}" for i in result.new_items if i.type == "tool_call_item"])
print(result.final_output)

['find_customer{"name":"Helena Holý"}', 'get_last_invoice{"customer_id":6}']
Helena Holý’s most recent invoice is **#404**, dated **November 13, 2013**, for **$25.86**. I can’t delete invoices with the available tools.


The agent looked up Helena and her last invoice (404, $25.86) and said it cannot delete invoices with
the available tools: none of the five can write. A narrow tool answered the question the free SQL tool answered in lab 15, without giving the model a way
to write any SQL.

## Task 2: Let a visitor start over

`clear_session()` deletes every item stored under one session id. The page function checks for the exact words
`start over` before anything reaches the agent, so the model never decides whether to forget.

The setup cell from lab 15 Part D comes first: this notebook runs the agent in its own cells (Task 1) and then on a
page, which runs on another event loop, so the SDK gets an HTTP client that does not keep connections open.

In [8]:
import httpx
from agents import set_default_openai_client
from openai import AsyncOpenAI, DefaultAsyncHttpxClient

set_default_openai_client(AsyncOpenAI(http_client=DefaultAsyncHttpxClient(
    limits=httpx.Limits(max_keepalive_connections=0))))

In [9]:
SESSIONS_DB = OUTPUTS / "solution_sessions.db"
SESSIONS_DB.unlink(missing_ok=True)


async def respond_start_over(message, history, request: gr.Request):
    session = SQLiteSession(request.session_hash, SESSIONS_DB)
    if message.strip() == "start over":
        await session.clear_session()
        return "New conversation."
    result = await Runner.run(desk_agent, message, session=session)
    return result.final_output

Two visitors. A `gradio_client.Client` knows its own `session_hash`, so after the messages we can read each visitor's
conversation from the sessions file, with `get_items()`.

In [10]:
from gradio_client import Client

ASK_A = "How many invoices does Helena Holý have, and who is her support rep?"
ASK_B = "Which customer were we talking about?"

page = gr.ChatInterface(respond_start_over)
page.launch(server_name="127.0.0.1", server_port=7860, prevent_thread_lock=True, inline=False, quiet=True)
visitor_a, visitor_b = Client(page.local_url, verbose=False), Client(page.local_url, verbose=False)
api = "/respond_start_over"

print("A:", visitor_a.predict(ASK_A, api_name=api))
print("B:", visitor_b.predict(ASK_A, api_name=api))
a_before = len(await SQLiteSession(visitor_a.session_hash, SESSIONS_DB).get_items())
print("A:", visitor_a.predict("start over", api_name=api))
print("A:", visitor_a.predict(ASK_B, api_name=api))
page.close()

A: Helena Holý has 7 invoices, and her support rep is Steve Johnson.


B: Helena Holý has 7 invoices, and her support rep is Steve Johnson.
A: New conversation.


A: I’m not sure which customer you mean. Could you share their name or another detail?
Closing server running on port: 7860


In [11]:
a_items = await SQLiteSession(visitor_a.session_hash, SESSIONS_DB).get_items()
b_items = await SQLiteSession(visitor_b.session_hash, SESSIONS_DB).get_items()
print("A's items before 'start over':", a_before)
print("A's items now:", len(a_items), "| user messages:", [i["content"] for i in a_items if i.get("role") == "user"])
print("B's items now:", len(b_items), "| user messages:", [i["content"] for i in b_items if i.get("role") == "user"])

A's items before 'start over': 8
A's items now: 4 | user messages: ['Which customer were we talking about?']
B's items now: 8 | user messages: ['How many invoices does Helena Holý have, and who is her support rep?']


Visitor A's last question got *"I'm not sure which customer you mean..."*: after `start over` the agent
had nothing about Helena. In the sessions file, A's conversation went from 8 items to 4, and the only user message
left is the question asked after `start over` (the command itself never reached the agent, so it was not stored).
Visitor B's conversation still holds its 8 items and its Helena question.

Clearing happens in our code, on an exact command, and only for the session that sent it: visitor B's conversation is
untouched. A real page would put the same call behind a button.